# Which Clusters Change the Most from Init to Trained MFA?

Companion to `kmeans_init_vs_mfa_comparison.ipynb`, focused on one question:
**per cluster, how many samples change membership between the k-means init
partition and the trained MFA partition?**

Inputs (same token stream, same K, identity cluster correspondence):

- **k-means (init)**: `kmeans_centroid_assignments.pt` — nearest Euclidean
  centroid per token.
- **MFA (trained)**: `mfa_model_assignments.pt` — argmax-responsibility
  component per token.

For each cluster `k` we count, from the K×K contingency matrix:

- `left_k`   = samples in k-means cluster k that end up elsewhere under MFA
- `joined_k` = samples in MFA cluster k that came from another init cluster
- `churn_k`  = `left_k + joined_k` — the **membership-change count**
  (size of the symmetric difference `km_k Δ mfa_k`)
- `net_delta_k` = `mfa_size_k − km_size_k` — the net size change

and plot the distributions of these deltas, raw and normalized by cluster size.


## 1. Setup and Artifact Validation

In [1]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception as exc:
    px = None
    go = None
    print(f"Plotly unavailable: {exc}")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/dalg").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO = find_repo_root()

# ---------------------------------------------------------------- parameters
LAYER = 5
K = 1000
Q = 10
EPOCHS = 1
# ------------------------------------------------- paths built from parameters
MFA_RUN = REPO / "dalg-cache/pile_gemma2b_activations/layer05_1000_10_mfa_1epoch_20260703_1538"
MFA_ASSIGN_PATH = MFA_RUN / "mfa_model_assignments.pt"
MFA_INIT_CENTROIDS = MFA_RUN / "centroids.pt"

CENTROIDS_DIR = REPO / f"dalg-cache/pile_gemma2b_activations/centroids/k{K}_L{LAYER:02d}"
KMEANS_CENTROIDS = CENTROIDS_DIR / "centroids.pt"
KMEANS_ASSIGN = CENTROIDS_DIR / "kmeans_centroid_assignments.pt"

# ------------------------------------------------------------- plot constants
COLOR_KMEANS = "#2a78d6"  # blue  — k-means init partition
COLOR_MFA = "#1baf7a"     # aqua  — trained MFA partition
COLOR_NEUTRAL = "#52514e" # gray  — derived quantities (deltas)
PLOT_TEMPLATE = "plotly_white"

PLOTS_DIR = REPO / "notebooks/plots"
PLOTS_HTML_DIR = PLOTS_DIR / "html"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_HTML_DIR.mkdir(parents=True, exist_ok=True)
PLOT_TAG = f"L{LAYER:02d}_K{K}_q{Q}_ep{EPOCHS}_membership"


def save_fig(fig, name: str) -> None:
    fig.update_layout(title_font_size=13, title_x=0.5, margin=dict(t=48))
    path = PLOTS_DIR / f"{name}_{PLOT_TAG}.pdf"
    try:
        fig.write_image(str(path), width=1000, height=550)
    except Exception as exc:
        path = PLOTS_HTML_DIR / f"{name}_{PLOT_TAG}.html"
        fig.write_html(str(path), include_plotlyjs="cdn")
        print(f"PDF export failed ({exc}); saved {path.name} instead")


artifacts = pd.DataFrame(
    [
        {"artifact": "kmeans assignments", "path": str(KMEANS_ASSIGN), "exists": KMEANS_ASSIGN.exists()},
        {"artifact": "mfa assignments", "path": str(MFA_ASSIGN_PATH), "exists": MFA_ASSIGN_PATH.exists()},
        {"artifact": "kmeans centroids", "path": str(KMEANS_CENTROIDS), "exists": KMEANS_CENTROIDS.exists()},
        {"artifact": "mfa init centroids", "path": str(MFA_INIT_CENTROIDS), "exists": MFA_INIT_CENTROIDS.exists()},
    ]
)
display(artifacts)
assert KMEANS_ASSIGN.exists() and MFA_ASSIGN_PATH.exists(), "missing assignment artifacts"


,artifact,path,exists
0,kmeans assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
1,mfa assignments,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
2,kmeans centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True
3,mfa init centroids,/orfeo/cephfs/home/dssc/zenocosini/decomposing...,True


### 1.1 Cluster Correspondence Check

Per-cluster deltas only make sense if cluster id `k` means the same thing in
both partitions, i.e. the nearest-centroid assignments used the exact centroids
the MFA was initialized from. Check bit-identity; if it fails, do not trust the
identity mapping below.


In [2]:
def _centroid_tensor(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict):
        for key in ("centroids", "mu", "means"):
            if key in obj:
                obj = obj[key]
                break
    return obj.float()


if KMEANS_CENTROIDS.exists() and MFA_INIT_CENTROIDS.exists():
    c_km = _centroid_tensor(KMEANS_CENTROIDS)
    c_init = _centroid_tensor(MFA_INIT_CENTROIDS)
    identity_ok = c_km.shape == c_init.shape and torch.equal(c_km, c_init)
    print(f"kmeans centroids: {tuple(c_km.shape)}, mfa init centroids: {tuple(c_init.shape)}")
    print(f"bit-identical: {identity_ok}")
    if not identity_ok:
        print("WARNING: centroids differ -> per-cluster deltas below are NOT meaningful under identity mapping.")
    del c_km, c_init
else:
    print("Skipped: missing centroid file(s); identity correspondence UNVERIFIED.")


kmeans centroids: (1000, 2048), mfa init centroids: (1000, 2048)
bit-identical: True


## 2. Load Assignments and Build the Contingency Matrix

Everything per-cluster derives from the K×K contingency matrix
`C[i, j] = #tokens with kmeans id i and mfa id j`, computed in one `bincount`
pass so both ~N-length assignment vectors can be freed immediately.


In [3]:
def load_assignments(path: Path) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    assert int(obj["K"]) == K, (obj["K"], K)
    return obj["assignments"].to(torch.long)


a_km = load_assignments(KMEANS_ASSIGN)
a_mfa = load_assignments(MFA_ASSIGN_PATH)
assert a_km.numel() == a_mfa.numel(), (a_km.numel(), a_mfa.numel())
N = a_km.numel()
print(f"Loaded {N:,} token assignments for both partitions")

contingency = (
    torch.bincount(a_km * K + a_mfa, minlength=K * K).reshape(K, K).numpy().astype(np.int64)
)
del a_km, a_mfa
gc.collect()

km_sizes = contingency.sum(axis=1)
mfa_sizes = contingency.sum(axis=0)
shared = np.diag(contingency).copy()

same_id_agreement = shared.sum() / N
print(f"same-id agreement: {same_id_agreement:.4f} "
      f"({N - shared.sum():,} of {N:,} tokens changed cluster)")


Loaded 73,687,936 token assignments for both partitions
same-id agreement: 0.5681 (31,823,784 of 73,687,936 tokens changed cluster)


### 2.1 Cluster Size Distribution

Compare the number of tokens assigned to each cluster before and after MFA
training. Both distributions use the same log-spaced bins.


In [4]:
if go is not None:
    positive_sizes = np.concatenate([km_sizes[km_sizes > 0], mfa_sizes[mfa_sizes > 0]])
    edges = np.logspace(np.log10(positive_sizes.min()), np.log10(positive_sizes.max()), 61)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)

    fig = go.Figure()
    for sizes, name, color in [
        (km_sizes, "k-means (init)", COLOR_KMEANS),
        (mfa_sizes, "MFA (trained)", COLOR_MFA),
    ]:
        counts, _ = np.histogram(sizes[sizes > 0], bins=edges)
        fig.add_bar(x=centers, y=counts, width=widths, name=name, marker_color=color, opacity=0.6)

    fig.update_xaxes(type="log")
    fig.update_layout(
        title="Cluster size distribution",
        xaxis_title="assigned tokens per cluster (log-spaced bins)",
        yaxis_title="clusters",
        template=PLOT_TEMPLATE,
        barmode="overlay",
        bargap=0.05,
        width=1000,
        height=550,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    )
    save_fig(fig, "cluster_size_hist")
    fig.show()

    print(f"Zero-size clusters: k-means={(km_sizes == 0).sum()}, MFA={(mfa_sizes == 0).sum()}")


Zero-size clusters: k-means=0, MFA=0


## 3. Per-Cluster Membership Change

For each cluster id `k` (identity mapping):

- `left = km_size − shared`: tokens that were in init cluster k but are
  assigned elsewhere by the trained MFA;
- `joined = mfa_size − shared`: tokens the trained cluster k gained from other
  init clusters;
- `churn = left + joined`: total membership changes touching cluster k;
- `net_delta = mfa_size − km_size = joined − left`: net growth/shrinkage;
- normalized versions divide by the init cluster size (`rel_*`), so a
  `rel_churn` of 1 means as many tokens moved as the cluster originally held.


In [5]:
per_cluster = pd.DataFrame(
    {
        "cluster": np.arange(K),
        "km_size": km_sizes,
        "mfa_size": mfa_sizes,
        "shared": shared,
    }
)
per_cluster["left"] = per_cluster["km_size"] - per_cluster["shared"]
per_cluster["joined"] = per_cluster["mfa_size"] - per_cluster["shared"]
per_cluster["churn"] = per_cluster["left"] + per_cluster["joined"]
per_cluster["net_delta"] = per_cluster["mfa_size"] - per_cluster["km_size"]
denom = np.maximum(per_cluster["km_size"], 1)
per_cluster["rel_left"] = per_cluster["left"] / denom
per_cluster["rel_churn"] = per_cluster["churn"] / denom
per_cluster["rel_net_delta"] = per_cluster["net_delta"] / denom

display(
    per_cluster[["km_size", "mfa_size", "left", "joined", "churn", "net_delta", "rel_churn"]]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)


,km_size,mfa_size,left,joined,churn,net_delta,rel_churn
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,73687.936000,73687.936000,31823.784000,31823.784000,63647.568000,0.000000,0.978088
std,57194.461082,55967.841465,36696.071622,30064.969461,58524.077185,32801.532161,0.692103
min,413.000000,494.000000,0.000000,6.000000,81.000000,-202605.000000,0.006086
10%,22326.100000,29905.700000,1925.200000,6319.200000,11751.300000,-38525.000000,0.222625
25%,35702.750000,42635.500000,6002.750000,13652.500000,24143.000000,-11667.250000,0.507945
50%,56763.500000,59215.000000,18678.000000,25667.000000,44905.500000,4087.000000,0.911723
75%,92937.500000,84424.250000,44716.750000,38951.750000,85088.000000,15965.250000,1.274092
90%,147550.400000,135239.100000,83253.500000,60656.400000,139407.300000,28707.400000,1.714266
max,561332.000000,618833.000000,289898.000000,358467.000000,573222.000000,255878.000000,8.431629


### 3.1 Clusters That Change the Most

Top movers by raw churn (dominated by big clusters) and by relative churn
(membership turnover regardless of size).


In [6]:
cols = ["cluster", "km_size", "mfa_size", "shared", "left", "joined", "churn", "net_delta", "rel_churn"]
display(Markdown("**Top 15 by raw churn (samples changing membership):**"))
display(per_cluster.sort_values("churn", ascending=False)[cols].head(15))

display(Markdown("**Top 15 by relative churn (churn / init size):**"))
display(per_cluster.sort_values("rel_churn", ascending=False)[cols].head(15))


**Top 15 by raw churn (samples changing membership):**

,cluster,km_size,mfa_size,shared,left,joined,churn,net_delta,rel_churn
383,383,561332,554758,271434,289898,283324,573222,-6574,1.021182
91,91,362955,618833,260366,102589,358467,461056,255878,1.270284
923,923,289206,122237,32804,256402,89433,345835,-166969,1.195809
72,72,292768,314048,135039,157729,179009,336738,21280,1.150187
610,610,200611,214632,41265,159346,173367,332713,14021,1.658498
27,27,356567,230701,128958,227609,101743,329352,-125866,0.923675
418,418,232435,253967,91820,140615,162147,302762,21532,1.302566
434,434,253425,115315,35754,217671,79561,297232,-138110,1.172860
711,711,242436,116325,40271,202165,76054,278219,-126111,1.147598
940,940,143989,234736,50276,93713,184460,278173,90747,1.931905


**Top 15 by relative churn (churn / init size):**

,cluster,km_size,mfa_size,shared,left,joined,churn,net_delta,rel_churn
813,813,4439,33661,336,4103,33325,37428,29222,8.431629
794,794,12641,68436,8034,4607,60402,65009,55795,5.142710
130,130,22487,83201,2254,20233,80947,101180,60714,4.499489
703,703,9181,31763,475,8706,31288,39994,22582,4.356170
360,360,27234,94775,1759,25475,93016,118491,67541,4.350848
564,564,20292,66395,911,19381,65484,84865,46103,4.182190
608,608,19218,83470,12281,6937,71189,78126,64252,4.065251
688,688,17857,56625,1387,16470,55238,71708,38768,4.015680
234,234,16497,50981,3603,12894,47378,60272,34484,3.653513
532,532,13478,47424,6291,7187,41133,48320,33946,3.585102


## 4. Distribution of the Membership-Change Delta

Histograms over the K clusters. `churn` is heavy-tailed, so it is shown on a
log-x axis alongside the size-normalized version; `net_delta` shows whether
training grew or shrank each cluster.


In [7]:
if px is not None:
    # px.histogram(log_x=True) bins linearly and only log-scales the axis, which
    # squashes everything into a few invisible bars; bin in log space instead.
    # churn, left and joined share bins so the three profiles are comparable.
    both = np.concatenate([per_cluster[c].to_numpy() for c in ("churn", "left", "joined")])
    both = np.clip(both, 1, None)
    edges = np.logspace(np.log10(both.min()), np.log10(both.max()), 61)
    centers = (edges[:-1] + edges[1:]) / 2
    widths = np.diff(edges)

    fig = go.Figure()
    for col, name, color in [
        ("churn", "churn (left + joined)", COLOR_NEUTRAL),
        ("left", "left (tokens lost)", COLOR_KMEANS),
        ("joined", "joined (tokens gained)", COLOR_MFA),
    ]:
        counts, _ = np.histogram(np.clip(per_cluster[col].to_numpy(), 1, None), bins=edges)
        fig.add_bar(x=centers, y=counts, width=widths, name=name, marker_color=color, opacity=0.6)
    fig.update_xaxes(type="log")
    fig.update_layout(
        title="Per-cluster membership change (churn = samples that left + samples that joined)",
        xaxis_title="tokens (log-spaced bins)",
        yaxis_title="clusters",
        template=PLOT_TEMPLATE,
        barmode="overlay",
        bargap=0.05,
        width=1000,
        height=600,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    )
    save_fig(fig, "churn_hist")
    fig.show()

    fig = px.histogram(
        per_cluster,
        x="rel_churn",
        nbins=60,
        title="Per-cluster relative membership change (churn / init cluster size)",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
    )
    fig.add_vline(x=1.0, line_dash="dash", line_color="#0b0b0b",
                  annotation_text="churn = init size", annotation_position="top right")
    fig.update_layout(xaxis_title="relative churn", yaxis_title="clusters", bargap=0.05)
    save_fig(fig, "rel_churn_hist")
    fig.show()

    fig = px.histogram(
        per_cluster,
        x="net_delta",
        nbins=80,
        title="Per-cluster net size change (mfa size − kmeans size)",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
    )
    fig.add_vline(x=0, line_dash="dash", line_color="#0b0b0b")
    fig.update_layout(xaxis_title="net Δ size (tokens)", yaxis_title="clusters", bargap=0.05)
    save_fig(fig, "net_delta_hist")
    fig.show()


### 4.1 Leavers vs Joiners

Each point is a cluster: x = tokens it lost, y = tokens it gained. Points on
the diagonal exchanged members without changing size; far off-diagonal points
were substantially grown or drained by training.


In [8]:
if px is not None:
    fig = px.scatter(
        per_cluster,
        x="left",
        y="joined",
        hover_data=["cluster", "km_size", "mfa_size", "churn", "net_delta"],
        title="Per-cluster tokens lost vs gained (init → trained)",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
        opacity=0.55,
    )
    lim = float(max(per_cluster["left"].max(), per_cluster["joined"].max())) * 1.05
    fig.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim,
                  line=dict(color="#0b0b0b", dash="dash", width=1))
    fig.update_layout(xaxis_title="tokens that left cluster k", yaxis_title="tokens that joined cluster k")
    save_fig(fig, "left_vs_joined")
    fig.show()

    fig = px.scatter(
        per_cluster,
        x="km_size",
        y="rel_churn",
        log_x=True,
        hover_data=["cluster", "mfa_size", "churn", "net_delta"],
        title="Relative churn vs init cluster size",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
        opacity=0.55,
    )
    fig.update_layout(xaxis_title="init cluster size (log)", yaxis_title="relative churn")
    save_fig(fig, "rel_churn_vs_size")
    fig.show()


### 4.2 Init → Trained Flow Heatmap

Cell `(i, j)` counts the tokens that **left** k-means cluster `i` and
**joined** MFA cluster `j` — i.e. the off-diagonal contingency entry
`C[i, j]`. The diagonal (tokens that stayed in their cluster) is zeroed so it
does not swamp the color scale, and counts are shown as `log10(tokens + 1)`.
Hot rows are init clusters that training drained broadly; hot columns are
trained clusters that absorbed tokens from many init clusters.


In [9]:
# if go is not None:
#     flow = contingency.astype(np.float32).copy()
#     np.fill_diagonal(flow, 0.0)  # diagonal = retained tokens, not membership change
#     fig = go.Figure(
#         go.Heatmap(
#             z=np.log10(flow + 1.0),
#             colorscale="Viridis",
#             zmin=0,
#             zmax=4,
#             colorbar=dict(title="log10(tokens + 1)"),
#             hovertemplate="kmeans i=%{y}<br>mfa j=%{x}<br>log10(tokens+1)=%{z:.2f}<extra></extra>",
#         )
#     )
#     fig.update_layout(
#         title="Membership flow: tokens leaving k-means cluster i and joining MFA cluster j",
#         xaxis_title="MFA cluster j (trained)",
#         yaxis_title="k-means cluster i (init)",
#         yaxis=dict(autorange="reversed"),
#         template=PLOT_TEMPLATE,
#         width=820,
#         height=780,
#     )
#     save_fig(fig, "flow_heatmap")
#     fig.show()
#     del flow


### 4.3 Clustered Flow Heatmap (clustermap)

Same matrix as 4.2, but rows and columns are reordered by hierarchical
clustering (average linkage, Euclidean distance on the `log10(tokens+1)`
flow profiles) so that init clusters draining to similar destinations sit
next to each other, and trained clusters absorbing from similar sources sit
next to each other. Axis positions no longer correspond to cluster ids —
hover to see the true `(i, j)` pair. Blocks of bright cells are groups of
init clusters whose tokens training redistributed into a common set of
trained clusters.


In [10]:
# if go is not None:
#     from scipy.cluster.hierarchy import leaves_list, linkage

#     flow = contingency.astype(np.float32).copy()
#     np.fill_diagonal(flow, 0.0)
#     log_flow = np.log10(flow + 1.0)

#     # euclidean rather than cosine: clusters with zero outflow/inflow produce
#     # all-zero profiles, for which cosine distance is undefined
#     row_order = leaves_list(linkage(log_flow, method="average", metric="euclidean"))
#     col_order = leaves_list(linkage(log_flow.T, method="average", metric="euclidean"))
#     z = log_flow[np.ix_(row_order, col_order)]

#     fig = go.Figure(
#         go.Heatmap(
#             z=z,
#             x=[str(j) for j in col_order],
#             y=[str(i) for i in row_order],
#             colorscale="Viridis",
#             zmin=0,
#             zmax=4,
#             colorbar=dict(title="log10(tokens + 1)"),
#             hovertemplate="kmeans i=%{y}<br>mfa j=%{x}<br>log10(tokens+1)=%{z:.2f}<extra></extra>",
#         )
#     )
#     fig.update_layout(
#         title="Clustered membership flow (rows/cols reordered by hierarchical clustering)",
#         xaxis=dict(title="MFA cluster j (trained, clustered order)", showticklabels=False),
#         yaxis=dict(title="k-means cluster i (init, clustered order)", showticklabels=False, autorange="reversed"),
#         template=PLOT_TEMPLATE,
#         width=820,
#         height=780,
#     )
#     save_fig(fig, "flow_clustermap")
#     fig.show()
#     del flow, log_flow, z


## 4.4 Tokens Exchanged vs Centroid Distance

Does a cluster exchange more tokens with clusters that are near it, or far from
it? Each point below is a cluster; we correlate **how many tokens it exchanges**
against the **token-weighted mean centroid distance to its exchange partners**.
Each partner is weighted by the number of tokens actually exchanged with it
(`Σ_p C[k,p]·d(k,p) / Σ_p C[k,p]`), so the y-axis reflects where the token
*mass* goes — not a rare 1-token leak to a distant cluster — and is on the same
token-weighted footing as the x-axis. Stratified into:

- **give**: tokens `k` gave away (`left`) vs distance to partners `j`, weighted by `C[k,j]`;
- **take**: tokens `k` took in (`joined`) vs distance to partners `i`, weighted by `C[i,k]`;
- **give+take**: total `churn` vs distance to all partners, weighted by `C[k,p]+C[p,k]`.

The first cell computes the weighted mean partner distances (under identity
cluster correspondence, with both the **k-means centroids** and the **MFA
means** as cluster centers); the scatter cell plots tokens exchanged (x) against
that distance (y) with a Spearman ρ per stratum.


In [11]:
from scipy.spatial.distance import cdist

from dalg.models.mfa import load_mfa

# cluster centers under identity correspondence
c_km = _centroid_tensor(KMEANS_CENTROIDS).numpy()
_model = load_mfa(MFA_RUN / "mfa_model.pt", map_location="cpu")
mu = _model.mu.detach().float().cpu().numpy()
del _model
gc.collect()
assert c_km.shape == mu.shape == (K, c_km.shape[1])

# off-diagonal flow -> per-partner token weights (match the x-axis token counts)
_flow = contingency.astype(np.float64).copy()
np.fill_diagonal(_flow, 0.0)
W_give = _flow                 # W_give[k, j] = tokens k -> j        (rowsum = left)
W_take = _flow.T               # W_take[k, i] = tokens i -> k        (rowsum = joined)
W_both = W_give + W_take       # total tokens exchanged with partner (rowsum = churn)


def weighted_partner_mean(values: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Token-weighted mean of values[k, p] over partners p (weight = tokens exchanged)."""
    denom = weights.sum(axis=1)
    num = (values * weights).sum(axis=1)
    return np.where(denom > 0, num / np.maximum(denom, 1), np.nan)


exchange = pd.DataFrame({"cluster": np.arange(K)})
exchange["n_give"] = (W_give > 0).sum(axis=1)
exchange["n_take"] = (W_take > 0).sum(axis=1)
exchange["n_both"] = (W_both > 0).sum(axis=1)
for label, centers in [("km", c_km), ("mfa", mu)]:
    D = cdist(centers, centers)
    exchange[f"give_dist_{label}"] = weighted_partner_mean(D, W_give)
    exchange[f"take_dist_{label}"] = weighted_partner_mean(D, W_take)
    exchange[f"both_dist_{label}"] = weighted_partner_mean(D, W_both)

per_cluster = per_cluster.merge(exchange, on="cluster")

display(
    exchange[
        ["n_give", "n_take", "n_both",
         "give_dist_km", "take_dist_km", "both_dist_km",
         "give_dist_mfa", "take_dist_mfa", "both_dist_mfa"]
    ].describe(percentiles=[0.1, 0.5, 0.9])
)


,n_give,n_take,n_both,give_dist_km,take_dist_km,both_dist_km,give_dist_mfa,take_dist_mfa,both_dist_mfa
count,1000.000000,1000.00000,1000.00000,996.000000,1000.000000,1000.000000,996.000000,1000.000000,1000.000000
mean,214.367000,214.36700,323.99000,32.679462,32.212578,32.470353,35.864151,35.711146,35.790649
std,165.148019,133.37621,175.42005,9.760840,10.200740,10.170458,9.571976,10.371052,10.285833
min,0.000000,1.00000,1.00000,11.703835,11.658330,12.114612,9.298686,11.375000,10.945573
10%,43.000000,45.00000,99.90000,23.961668,22.002369,23.656508,26.611482,24.831181,25.623340
50%,158.000000,216.00000,318.00000,30.625628,30.496376,30.485553,34.695910,34.568965,34.472754
90%,483.000000,373.10000,568.10000,44.436728,44.128467,43.926195,46.033184,47.683687,46.629011
max,716.000000,774.00000,880.00000,130.446338,96.529675,121.468997,129.418859,106.341023,122.109677


In [12]:
if px is not None:
    from scipy.stats import spearmanr

    # per-cluster: x = tokens exchanged, y = weighted mean centroid distance to partners
    strata = [
        ("give", "left", COLOR_KMEANS),        # tokens k gave away
        ("take", "joined", COLOR_MFA),         # tokens k took in
        ("give+take", "churn", COLOR_NEUTRAL), # both
    ]
    for label, title in [("km", "k-means centroid geometry"), ("mfa", "MFA mean geometry")]:
        fig = go.Figure()
        for name, xcol, color in strata:
            ycol = f"{'both' if name == 'give+take' else name}_dist_{label}"
            sub = per_cluster[["cluster", xcol, ycol]].dropna()
            sub = sub[sub[xcol] > 0]
            x, y, cl = sub[xcol].to_numpy(), sub[ycol].to_numpy(), sub["cluster"].to_numpy()
            rho, _ = spearmanr(x, y)
            fig.add_scatter(
                x=x, y=y, mode="markers",
                name=f"{name} (ρ={rho:.2f})",
                marker=dict(color=color, size=5, opacity=0.5),
                customdata=cl,
                hovertemplate="cluster %{customdata}<br>exchanged=%{x:,}<br>distance=%{y:.2f}<extra></extra>",
            )
            # least-squares fit y ~ a*log10(x) + b (linear in the plotted log-x axis)
            lx = np.log10(x)
            a, b = np.polyfit(lx, y, 1)
            xs = np.linspace(lx.min(), lx.max(), 100)
            fig.add_scatter(
                x=10 ** xs, y=a * xs + b, mode="lines",
                line=dict(color=color, width=2),
                name=f"{name} fit", showlegend=False, hoverinfo="skip",
            )
        fig.update_xaxes(type="log")
        fig.update_layout(
            title=f"Tokens exchanged vs mean centroid distance to partners ({title})",
            xaxis_title="tokens exchanged (log)",
            yaxis_title="mean centroid distance to exchange partners",
            template=PLOT_TEMPLATE,
            width=1000,
            height=550,
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )
        save_fig(fig, f"exchange_distance_vs_count_{label}")
        fig.show()


## 4.5 Tokens Exchanged vs Intrinsic-Dimension Gap

Same construction as 4.4, but the y-axis replaces centroid distance with the
token-weighted mean **intrinsic-dimension gap** to the exchange partners: for
cluster `k`, `Σ_p C[k,p]·(ID[p] − ID[k]) / Σ_p C[k,p]` over its partners `p`
(each weighted by tokens exchanged). A positive value means `k`'s exchanged
tokens go mostly to/from clusters of *higher* intrinsic dimension than itself.

Stratified the same way — **give** vs `left`, **take** vs `joined`,
**give+take** vs `churn` (same token weights as 4.4) — and computed for both
intrinsic-dimension sources: the **k-means partition** IDs
(`centroids_1000_05/intrinsic_dims.pt`) and the **MFA partition** IDs
(`1000_05_10/intrinsic_dims.pt`), under identity cluster correspondence.


In [13]:
# intrinsic-dim per cluster from each partition (identity cluster correspondence)
ID_PATHS = {
    "km": REPO / "dalg-cache/output/experiments/centroids_1000_05/intrinsic_dims.pt",
    "mfa": REPO / "dalg-cache/output/experiments/1000_05_10/intrinsic_dims.pt",
}
ids_by_source = {}
for label, path in ID_PATHS.items():
    obj = torch.load(path, map_location="cpu")
    assert int(obj["K"]) == K, (label, obj["K"], K)
    ids_by_source[label] = obj["intrinsic_dims"].to(torch.float64).numpy()


def weighted_partner_id_gap(ids: np.ndarray, weights: np.ndarray) -> np.ndarray:
    """Token-weighted mean over partners of (partner_ID - k_ID); ID==0 = invalid partner."""
    valid = ids > 0
    diff = ids[None, :] - ids[:, None]     # diff[k, p] = ID[p] - ID[k]
    w = weights * valid[None, :]           # drop invalid partners from the weighting
    denom = w.sum(axis=1)
    num = (diff * w).sum(axis=1)
    return np.where((denom > 0) & valid, num / np.maximum(denom, 1), np.nan)


# W_give / W_take / W_both come from the 4.4 exchange cell (token weights)
for label, ids in ids_by_source.items():
    per_cluster[f"give_iddiff_{label}"] = weighted_partner_id_gap(ids, W_give)
    per_cluster[f"take_iddiff_{label}"] = weighted_partner_id_gap(ids, W_take)
    per_cluster[f"both_iddiff_{label}"] = weighted_partner_id_gap(ids, W_both)

display(
    per_cluster[
        ["give_iddiff_km", "take_iddiff_km", "both_iddiff_km",
         "give_iddiff_mfa", "take_iddiff_mfa", "both_iddiff_mfa"]
    ].describe(percentiles=[0.1, 0.5, 0.9])
)


FileNotFoundError: [Errno 2] No such file or directory: '/orfeo/cephfs/home/dssc/zenocosini/decomposing-activations-local-geometry/dalg-cache/output/experiments/1000_05_10/intrinsic_dims.pt'

In [ ]:
if px is not None:
    from scipy.stats import spearmanr

    strata = [
        ("give", "left", COLOR_KMEANS),
        ("take", "joined", COLOR_MFA),
        ("give+take", "churn", COLOR_NEUTRAL),
    ]
    for label, title in [
        ("km", "k-means partition intrinsic dims"),
        ("mfa", "MFA partition intrinsic dims"),
    ]:
        fig = go.Figure()
        for name, xcol, color in strata:
            ycol = f"{'both' if name == 'give+take' else name}_iddiff_{label}"
            sub = per_cluster[["cluster", xcol, ycol]].dropna()
            sub = sub[sub[xcol] > 0]
            x, y, cl = sub[xcol].to_numpy(), sub[ycol].to_numpy(), sub["cluster"].to_numpy()
            rho, _ = spearmanr(x, y)
            fig.add_scatter(
                x=x, y=y, mode="markers",
                name=f"{name} (ρ={rho:.2f})",
                marker=dict(color=color, size=5, opacity=0.5),
                customdata=cl,
                hovertemplate="cluster %{customdata}<br>exchanged=%{x:,}<br>ID gap=%{y:.1f}<extra></extra>",
            )
            # least-squares fit y ~ a*log10(x) + b (linear in the plotted log-x axis)
            lx = np.log10(x)
            a, b = np.polyfit(lx, y, 1)
            xs = np.linspace(lx.min(), lx.max(), 100)
            fig.add_scatter(
                x=10 ** xs, y=a * xs + b, mode="lines",
                line=dict(color=color, width=2),
                name=f"{name} fit", showlegend=False, hoverinfo="skip",
            )
        fig.add_hline(y=0, line_dash="dash", line_color="#0b0b0b")
        fig.update_xaxes(type="log")
        fig.update_layout(
            title=f"Tokens exchanged vs mean intrinsic-dim gap to partners ({title})",
            xaxis_title="tokens exchanged (log)",
            yaxis_title="mean (partner ID − cluster ID) over exchange partners",
            template=PLOT_TEMPLATE,
            width=1000,
            height=550,
            legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
        )
        save_fig(fig, f"exchange_iddiff_vs_count_{label}")
        fig.show()


## 4.6 Spectrum of a Single Cluster (401)

Cumulative fraction of within-cluster variance vs principal-component rank for
cluster 401, from the PCA spectra stored in `cluster_variances`. The k-means
curve and the MFA curve are overlaid: a curve that rises faster concentrates its
variance in fewer directions (lower intrinsic dimension). The dashed line marks
the variance threshold used to define the reported intrinsic dimension.


In [ ]:
CLUSTER_ID = 401
if go is not None:
    fig = go.Figure()
    vt = None
    for label, color, pretty in [("km", COLOR_KMEANS, "k-means"), ("mfa", COLOR_MFA, "MFA")]:
        obj = torch.load(ID_PATHS[label], map_location="cpu")
        v = obj["cluster_variances"][CLUSTER_ID].float().numpy()
        v = v[v > 0]
        cum = np.cumsum(v) / v.sum()
        ranks = np.arange(1, len(cum) + 1)
        id_val = int(obj["intrinsic_dims"][CLUSTER_ID])
        vt = float(obj["variance_threshold"])
        fig.add_scatter(
            x=ranks, y=cum, mode="lines",
            name=f"{pretty} (ID@{vt:.0%} = {id_val}, {len(cum)} PCs)",
            line=dict(color=color, width=2),
        )
    fig.add_hline(y=vt, line_dash="dash", line_color="#0b0b0b",
                  annotation_text=f"{vt:.0%} variance", annotation_position="bottom right")
    fig.update_layout(
        title=f"Cumulative explained-variance spectrum — cluster {CLUSTER_ID}",
        xaxis_title="principal component rank",
        yaxis_title="cumulative fraction of within-cluster variance",
        template=PLOT_TEMPLATE,
        width=1000,
        height=550,
        legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
    )
    save_fig(fig, f"spectrum_cluster{CLUSTER_ID}")
    fig.show()


## 5. Summary

Compact table of the headline numbers.


In [ ]:
summary = pd.DataFrame(
    [
        {"metric": "same-id agreement", "value": same_id_agreement},
        {"metric": "tokens changing cluster", "value": int(N - shared.sum())},
        {"metric": "median per-cluster churn", "value": float(per_cluster["churn"].median())},
        {"metric": "median relative churn", "value": float(per_cluster["rel_churn"].median())},
        {"metric": "clusters with rel_churn > 1", "value": int((per_cluster["rel_churn"] > 1).sum())},
        {"metric": "clusters that grew (net_delta > 0)", "value": int((per_cluster["net_delta"] > 0).sum())},
        {"metric": "clusters that shrank (net_delta < 0)", "value": int((per_cluster["net_delta"] < 0).sum())},
        {"metric": "max churn cluster", "value": int(per_cluster.loc[per_cluster["churn"].idxmax(), "cluster"])},
    ]
)
display(summary)


,metric,value
0,same-id agreement,5.681276e-01
1,tokens changing cluster,3.182378e+07
2,median per-cluster churn,4.490550e+04
3,median relative churn,9.117235e-01
4,clusters with rel_churn > 1,4.240000e+02
5,clusters that grew (net_delta > 0),5.910000e+02
6,clusters that shrank (net_delta < 0),4.090000e+02
7,max churn cluster,3.830000e+02


## 6. First-10 Principal-Component Alignment

For each identity-matched cluster, compare the spans of its first ten
empirical PCs under the k-means and MFA partitions. If `U` and `V` are the
two orthonormal D x 10 bases, the singular values of `U.T @ V` are the cosines
of the ten principal angles. Small angles indicate aligned local variation.

The intrinsic-dimension artifacts provide the matching assignment paths and
sampling metadata, but retain PCA eigenvalues rather than eigenvectors. This
cell therefore re-samples the original activation stream and reconstructs the
PC bases. The bounded sample cap keeps this notebook practical while leaving
the exact SVD calculation unchanged.


In [14]:
from dalg.analysis.cluster_intrinsic_dim import (
    _choose_sample_positions,
    _collect_sampled_activations,
)
from dalg.data.shard_activations import ActivationBatchDataset


PC_COUNT = 10
PC_SAMPLES_PER_CLUSTER = 512
PC_SAMPLE_SEED = 0
PC_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PC_ID_PATHS = {
    "km": REPO / "dalg-cache/output/experiments/centroids_1000_05/intrinsic_dims.pt",
    "mfa": REPO / "dalg-cache/output/experiments/1000_05_q10/intrinsic_dims.pt",
}
pc_id_results = {
    label: torch.load(path, map_location="cpu", weights_only=False)
    for label, path in PC_ID_PATHS.items()
}

# The PCA directions are not saved in intrinsic_dims.pt; recover the exact
# assignment inputs recorded by each artifact before re-sampling activations.
pc_assignment_paths = {
    label: Path(result["assignments_path"])
    for label, result in pc_id_results.items()
}
missing_assignments = [
    f"{label}: {path}" for label, path in pc_assignment_paths.items() if not path.exists()
]
if missing_assignments:
    raise FileNotFoundError(
        "The saved intrinsic-dimension spectra do not contain PCA directions. "
        "Restore the assignment artifact(s) recorded in intrinsic_dims.pt before running this cell:\n"
        + "\n".join(missing_assignments)
    )

km_source = pc_id_results["km"]["assignment_metadata"]["source"]
PC_SHARD_DIR = Path(km_source["shard_dir"])
PC_LAYER = int(km_source["layer"])
PC_DROP_PREFIX = int(km_source["drop_prefix"])


def first_empirical_pcs(
    assignment_path: Path,
    expected_sizes: torch.Tensor,
    available_samples: int,
) -> list[torch.Tensor | None]:
    assignment_data = torch.load(assignment_path, map_location="cpu", weights_only=True)
    assignments = assignment_data["assignments"].to(torch.long)
    sizes = assignment_data["cluster_sizes"].to(torch.long)
    assert assignments.numel() == N, (assignment_path, assignments.numel(), N)
    assert torch.equal(sizes, expected_sizes.to(torch.long)), (
        "assignment cluster sizes do not match the intrinsic-dimension artifact",
        assignment_path,
    )

    max_samples = min(PC_SAMPLES_PER_CLUSTER, available_samples)
    positions, clusters, _ = _choose_sample_positions(
        assignments,
        sizes,
        K=K,
        max_samples=max_samples,
        min_population=PC_COUNT,
        seed=PC_SAMPLE_SEED,
    )
    loader = ActivationBatchDataset(
        PC_SHARD_DIR,
        PC_LAYER,
        batch_size=8192,
        drop_prefix=PC_DROP_PREFIX,
        dtype=torch.float16,
        shuffle_shards=False,
        shuffle_within_shard=False,
    )
    buffers = _collect_sampled_activations(
        loader,
        positions,
        clusters,
        K=K,
        num_expected_items=assignments.numel(),
        store_dtype=torch.float16,
    )

    pcs: list[torch.Tensor | None] = [None] * K
    for cluster, X in enumerate(buffers):
        if X is None or X.shape[0] < PC_COUNT:
            continue
        centered = X.to(PC_DEVICE, dtype=torch.float32)
        centered -= centered.mean(dim=0, keepdim=True)
        _, _, vh = torch.linalg.svd(centered, full_matrices=False)
        pcs[cluster] = vh[:PC_COUNT].T.cpu()

    del assignments, buffers
    gc.collect()
    if PC_DEVICE == "cuda":
        torch.cuda.empty_cache()
    return pcs


pc_bases = {
    label: first_empirical_pcs(
        pc_assignment_paths[label],
        result["cluster_sizes"],
        int(result["max_samples"]),
    )
    for label, result in pc_id_results.items()
}

alignment_rows = []
for cluster, (km_basis, mfa_basis) in enumerate(zip(pc_bases["km"], pc_bases["mfa"])):
    if km_basis is None or mfa_basis is None:
        continue
    cosines = torch.linalg.svdvals(km_basis.T @ mfa_basis).clamp(0, 1)
    angles_deg = torch.rad2deg(torch.acos(cosines)).numpy()
    row = {
        "cluster": cluster,
        "mean_angle_deg": float(angles_deg.mean()),
        "max_angle_deg": float(angles_deg.max()),
        "mean_squared_cosine": float((cosines.square()).mean()),
    }
    row.update({f"angle_{i + 1}_deg": float(angle) for i, angle in enumerate(angles_deg)})
    alignment_rows.append(row)

pc_alignment = pd.DataFrame(alignment_rows).sort_values("mean_angle_deg").reset_index(drop=True)
display(pc_alignment.describe(percentiles=[0.1, 0.5, 0.9]))
display(pc_alignment.head(10))

if px is not None:
    fig = px.histogram(
        pc_alignment,
        x="mean_angle_deg",
        nbins=50,
        title="Alignment of first-10 empirical PC subspaces: k-means vs MFA",
        color_discrete_sequence=[COLOR_NEUTRAL],
        template=PLOT_TEMPLATE,
    )
    fig.update_layout(
        xaxis_title="mean principal angle (degrees; lower = more aligned)",
        yaxis_title="clusters",
        bargap=0.05,
        width=1000,
        height=550,
    )
    save_fig(fig, "pc_alignment_hist")
    fig.show()


FileNotFoundError: The saved intrinsic-dimension spectra do not contain PCA directions. Restore the assignment artifact(s) recorded in intrinsic_dims.pt before running this cell:
mfa: /orfeo/scratch/dssc/zenocosini/dalg-cache/pile_gemma2b_activations/archived/layer05_1000_10_mfa/mfa_model_assignments.pt